# Exercises

There are three exercises in this notebook:

1. Use the cross-validation method to test the linear regression with different $\alpha$ values, at least three.
2. Implement a SGD method that will train the Lasso regression for 10 epochs.
3. Extend the Fisher's classifier to work with two features. Use the class as the $y$.

## 1. Cross-validation linear regression

You need to change the variable ``alpha`` to be a list of alphas. Next do a loop and finally compare the results.

In [5]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

x = np.array([188, 181, 197, 168, 167, 187, 178, 194, 140, 176, 168, 192, 173, 142, 176]).reshape(-1, 1)
y = np.array([141, 106, 149, 59, 79, 136, 65, 136, 52, 87, 115, 140, 82, 69, 121]).reshape(-1, 1)

alphas = [0.1, 1.0, 10.0]

for alpha in alphas:
    model = Ridge(alpha=alpha)
    scores = cross_val_score(model, x, y.ravel(), cv=5, scoring='neg_mean_squared_error')
    mse = -scores.mean()
    print("alpha", alpha, "mean squared error", mse)

alpha 0.1 mean squared error 468.6580797993185
alpha 1.0 mean squared error 468.6081330113103
alpha 10.0 mean squared error 468.1216295847045


## 2. Implement based on the Ridge regression example, the Lasso regression.

Please implement the SGD method and compare the results with the sklearn Lasso regression results. 

In [6]:
def sgd(x, y, alpha, learning_rate, epochs):
    n, m = x.shape
    w = np.zeros((m, 1))
    for epoch in range(epochs):
        for i in range(n):
            xi = x[i].reshape(1, -1)
            yi = y[i].reshape(1, -1)
            y_pred = xi.dot(w)
            error = y_pred - yi
            grad = xi.T.dot(error) + alpha * np.sign(w)
            w = w - learning_rate * grad
    return w

In [7]:
import numpy as np
from sklearn.linear_model import Lasso

x = np.array([188, 181, 197, 168, 167, 187, 178, 194, 140, 176, 168, 192, 173, 142, 176]).reshape(-1, 1)
y = np.array([141, 106, 149, 59, 79, 136, 65, 136, 52, 87, 115, 140, 82, 69, 121]).reshape(-1, 1)

x_mean = x.mean()
x_std = x.std()
x_norm = (x - x_mean) / x_std

x_bias = np.c_[np.ones((15, 1)), x_norm]

alpha = 0.1
learning_rate = 0.01
epochs = 10

w = sgd(x_bias, y, alpha, learning_rate, epochs)
print("weights from sgd lasso")
print(w.ravel())

y_pred_sgd = x_bias.dot(w)
mse_sgd = np.mean((y_pred_sgd - y) ** 2)
print("mean squared error sgd lasso", mse_sgd)

lasso_regression = Lasso(alpha=alpha)
lasso_regression.fit(X=x, y=y.ravel())
print("weights from sklearn lasso")
print(lasso_regression.intercept_, lasso_regression.coef_)

y_pred_lasso = lasso_regression.predict(x)
mse_lasso = np.mean((y_pred_lasso - y.ravel()) ** 2)
print("mean squared error sklearn lasso", mse_lasso)

weights from sgd lasso
[80.04380825 19.979521  ]
mean squared error sgd lasso 915.5366364276641
weights from sklearn lasso
-180.85790859980537 [1.61776499]
mean squared error sklearn lasso 372.33132989967453


## 3. Extend the Fisher's classifier

Please extend the targets of the ``iris_data`` variable and use it as the $y$.

In [8]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

iris_data = load_iris()
iris_df = pd.DataFrame(iris_data.data, columns=iris_data.feature_names)

x = iris_df[['sepal width (cm)', 'sepal length (cm)']].values
y = iris_data.target

x = x[y != 2]
y = y[y != 2]

mean_0 = np.mean(x[y == 0], axis=0)
mean_1 = np.mean(x[y == 1], axis=0)

diff_0 = x[y == 0] - mean_0
diff_1 = x[y == 1] - mean_1

Sw = diff_0.T.dot(diff_0) + diff_1.T.dot(diff_1)

w = np.linalg.inv(Sw).dot(mean_1 - mean_0)
w0 = 0.5 * (w.dot(mean_1) + w.dot(mean_0))

predictions = []
for xi in x:
    score = w.dot(xi) - w0
    if score > 0:
        predictions.append(1)
    else:
        predictions.append(0)

print("weights", w)
print("threshold", w0)
print("accuracy", accuracy_score(y, predictions))

weights [-0.14431661  0.11669751]
threshold 0.1912149157461091
accuracy 0.99
